In [0]:
df_prod = spark.read.format("csv")\
                    .option("header" , True)\
                    .option("inferSchema" , True)\
                    .load("/Volumes/ashlamba/bronze/bronze_volume/products/")
df_prod.display()

In [0]:
from pyspark.sql.functions import current_timestamp
df_prod = df_prod.withColumn("process_date" , current_timestamp())

In [0]:
df_prod.display()

In [0]:
from pyspark.sql.functions import col,avg
df_prod.groupBy(col("category")).agg(avg(col("price")).alias("avg_price")).display()

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import expr

if spark.catalog.tableExists("ashlamba.silver.product_enr"):
    from delta.tables import DeltaTable
    prod_obj = DeltaTable.forName(spark,"ashlamba.silver.product_enr")
    (
        prod_obj.alias("t").merge(df_prod.alias("s"),
                                  expr("t.product_id = s.product_id"))
                            .whenMatchedUpdateAll()
                            .whenNotMatchedInsertAll()
                            .execute()
    )

else:
    df_prod.write.format("delta")\
                .mode("append")\
                .saveAsTable("ashlamba.silver.product_enr")

In [0]:
%sql
select * from ashlamba.silver.product_enr;

In [0]:
df_str = spark.read.format("csv")\
                    .option("inferSchema" , True)\
                    .option("header" , True)\
                    .load('/Volumes/ashlamba/bronze/bronze_volume/stores/')
df_str.display()

In [0]:
from pyspark.sql.functions import reg

In [0]:
df_str = df_str.withColumn("store_name", regexp_replace(col("store_name"),"_" , ""))
df_str.display()

In [0]:
df_str = df_str.withColumn("process_date",current_timestamp())
df_str.display()

In [0]:
from pyspark.sql.functions import expr ,col 
from delta.tables import DeltaTable 

if spark.catalog.tableExists("ashlamba.silver.stores_enr"):
    store_obj = DeltaTable.forName(spark,'ashlamba.silver.stores_enr')
    (
        store_obj.alias("t").merge(df_str.alias("s"),
                                   expr("t.store_id = s.store_id"))
                            .whenMatchedUpdateAll()
                            .whenNotMatchedInsertAll()
                            .execute()
    )

else:
    df_str.write.format("delta")\
                .mode("append")\
                .saveAsTable("ashlamba.silver.stores_enr")

In [0]:
%sql
select * from ashlamba.silver.stores_enr;

In [0]:
df_sale = spark.read.format("csv")\
                    .option("header",True)\
                    .option("inferSchema",True)\
                    .load("/Volumes/ashlamba/bronze/bronze_volume/sales/")
df_sale.display()

In [0]:
from pyspark.sql.types import DecimalType
df_sale = df_sale.withColumns(
                              {
                                "price_per_sale" : round(col("total_amount").cast(DecimalType(10,2))/col("quantity").cast(DecimalType(10,2)),2 ),
                                "process_date" : current_timestamp()
                              }
                          )
df_sale.display()

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists("ashlamba.silver.sales_enr"):
    sales_obj = DeltaTable.forName(spark,"ashlamba.silver.sales_enr")

    (
        sales_obj.alias("t").merge(df_sale.alias("s"),expr("t.sales_id = s.sales_id"))
                                .whenMatchedUpdateAll()
                                .whenNotMatchedInsertAll()
                                .execute()
    )

else:
    df_sale.write.format("delta")\
            .mode("append")\
            .saveAsTable("ashlamba.silver.sales_enr")

In [0]:
%sql
select * from ashlamba.silver.sales_enr;